# CropGuard - Baseline Training

Runs on **Lightning AI Studio** or **Google Colab** - the setup cell detects which and
sets paths accordingly. Attach a GPU before running (T4 is plenty).

data -> validate -> split -> train -> ONNX -> INT8 -> holdout evaluation. ~90 minutes,
most of it in the training cell.

The dataset is pulled from HuggingFace here, not uploaded from your machine. The split is
regenerated from `seed: 42` and hash-checked, so results stay comparable across machines
without moving 2.2GB around.

> **Lightning AI note:** the studio filesystem is persistent, so checkpoints and data
> survive between sessions. Re-running this notebook after a break skips straight past the
> download, and training can resume from the last checkpoint.

## 1. Setup

Clones the repo, installs dependencies, and puts the package on the import path.

We deliberately do **not** `pip install -e .`. An editable install registers the package
through a `.pth` file, and `.pth` files are only read when the interpreter starts - so a
kernel that is already running cannot see it and `import cropguard` fails. Adding `src/`
to `sys.path` is simpler and works immediately.

In [ ]:
import os, sys, subprocess
from pathlib import Path

# Lightning AI studios live under /teamspace and persist; Colab is ephemeral under
# /content. Detect rather than hardcode so the same notebook runs on both.
if Path('/teamspace/studios/this_studio').exists():
    ENV, BASE = 'lightning', '/teamspace/studios/this_studio'
elif Path('/content').exists():
    ENV, BASE = 'colab', '/content'
else:
    ENV, BASE = 'local', str(Path.home())

REPO = f'{BASE}/CropGuard'
DATA = f'{BASE}/cropguard-data'
SRC  = f'{REPO}/src'

print('environment :', ENV)
print('repo        :', REPO)
print('data        :', DATA)
print('persistent  :', ENV == 'lightning')

In [ ]:
if Path(REPO + '/.git').exists():
    print('repo present - pulling latest')
    !git -C {REPO} pull -q
else:
    !rm -rf {REPO}
    !git clone -q https://github.com/abhinav7289A/CropGuard.git {REPO}

os.chdir(REPO)          # subprocesses inherit cwd, so every `!python -m ...` below works
!git log --oneline -1

In [ ]:
# Both platforms ship torch with CUDA. Installing '.[train]' would pull a second torch
# build and can silently replace it with a CPU wheel, so install only what is missing.
!pip install -q pytorch-lightning timm torchmetrics wandb onnx onnxruntime huggingface_hub scikit-learn tqdm pyyaml

In [ ]:
import importlib

if SRC not in sys.path:
    sys.path.insert(0, SRC)
importlib.invalidate_caches()

# Every pipeline step runs as `!python -m cropguard...` in a fresh subprocess, which does
# not inherit sys.path - PYTHONPATH is what makes those work.
os.environ['PYTHONPATH'] = SRC
os.environ['CROPGUARD_DATA_DIR'] = DATA
os.environ['PYTHONIOENCODING'] = 'utf-8'

import cropguard, torch
print('cropguard  ', cropguard.__version__, 'from', os.path.dirname(cropguard.__file__))
print('torch      ', torch.__version__, '| CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu        ', torch.cuda.get_device_name(0))
!python -c "import cropguard; print('subprocess OK  ', cropguard.__version__)"

assert torch.cuda.is_available(), 'No GPU - attach one to the studio / change Colab runtime'

## 2. Weights & Biases (optional)

Paste your key for experiment tracking, or leave blank to log to CSV.
Key: https://wandb.ai/authorize

**Leave `WANDB_ENTITY` empty unless you are logging to a team.** Your entity is not always
the same string as your login name, and setting it wrong fails with
`entity ... not found during upsertBucket`. Unset, W&B uses your account default.

If W&B fails for any reason, training falls back to CSV and carries on rather than dying.

In [ ]:
WANDB_API_KEY = ''   # <- paste your key
WANDB_ENTITY  = ''   # <- leave empty for a personal account; team name only

os.environ.pop('WANDB_ENTITY', None)

if WANDB_API_KEY:
    os.environ['WANDB_API_KEY'] = WANDB_API_KEY
    if WANDB_ENTITY:
        os.environ['WANDB_ENTITY'] = WANDB_ENTITY
    os.environ.pop('WANDB_MODE', None)
    import wandb
    wandb.login(key=WANDB_API_KEY)   # fail here, not 45 minutes into training
    print('W&B enabled | entity:', WANDB_ENTITY or '(account default)')
else:
    os.environ['WANDB_MODE'] = 'disabled'
    print('W&B disabled - CSV logging only')

## 3. Data

Downloads `data.zip` (2.2GB) from HuggingFace, extracts the `color` variant, validates it,
and builds the leaf-grouped split. ~20 minutes the first time.

On Lightning AI this is persistent - later sessions skip straight past it.

The split hash must print `MATCH`. If it does not, the data changed upstream and results
are not comparable to earlier runs.

In [ ]:
if Path(DATA + '/plantvillage/manifest.json').exists():
    import json
    m = json.load(open(DATA + '/plantvillage/manifest.json'))
    print(f"dataset already present: {m['num_images']:,} images / {m['num_classes']} classes")
else:
    !python -m cropguard.data.download --config configs/base.yaml

In [ ]:
!python -m cropguard.data.validate --config configs/base.yaml

In [ ]:
!python -m cropguard.data.split --config configs/base.yaml

import hashlib, json
EXPECTED = '9764d8f2eb2046e9ba91a138e21d472bd6a9e512232431b7d62d252c6ea8efba'
actual = hashlib.sha256(open(DATA + '/splits.json','rb').read()).hexdigest()
print()
print('split hash:', 'MATCH' if actual == EXPECTED else 'MISMATCH -> ' + actual)
print('leakage   :', json.load(open(DATA + '/split_report.json'))['leakage']['test'])

## 4. Checkpoint persistence

On Lightning AI the studio filesystem already persists - nothing to do. On Colab, sessions
are reclaimed without warning, so checkpoints are symlinked to Drive; without that, a drop
means starting over.

In [ ]:
if ENV == 'colab':
    from google.colab import drive
    drive.mount('/content/drive')
    !mkdir -p /content/drive/MyDrive/cropguard/checkpoints
    !ln -sfn /content/drive/MyDrive/cropguard/checkpoints {REPO}/checkpoints
    print('checkpoints -> Google Drive')
else:
    os.makedirs(f'{REPO}/checkpoints', exist_ok=True)
    print('checkpoints ->', f'{REPO}/checkpoints', '(persistent filesystem)')

## 5. Train

~45-60 min for 12 epochs on a T4, faster on an A10G. `num_workers` is set from the actual
CPU count - Colab gives ~2, Lightning studios usually more, and using all of them on Colab
would cause contention rather than throughput.

**If the session dies:** re-run cells 1-4, then add
`--resume checkpoints/resnet50-baseline/last.ckpt` to the training command.

**If you hit CUDA OOM:** lower `batch_size` in the config cell. Mention it when reporting
results - it changes the effective learning-rate schedule.

In [ ]:
workers = min(8, max(2, (os.cpu_count() or 2) - 1))
print('cpu count:', os.cpu_count(), '-> num_workers:', workers)

config = f'''# Generated for this runtime. Same baseline, workers matched to the machine.
extends: resnet50_baseline.yaml

data:
  num_workers: {workers}
'''
Path('configs/colab_resnet50.yaml').write_text(config)
print()
print(config)

In [ ]:
# 30-second smoke test - catches config and data errors before the long run.
!python -m cropguard.training.train --config configs/colab_resnet50.yaml --fast-dev-run

In [ ]:
!python -m cropguard.training.train --config configs/colab_resnet50.yaml

## 6. Export to ONNX + INT8

Gated by a PyTorch parity check (max |logit diff| < 1e-3): a silently wrong graph is the
worst failure mode here, so the export raises rather than shipping one.

In [ ]:
import glob
ckpts = sorted(glob.glob('checkpoints/resnet50-baseline/best-*.ckpt'))
assert ckpts, 'No checkpoint - did training finish?'
CKPT = ckpts[-1]
print('exporting', CKPT)

!python -m cropguard.serving.onnx_export --ckpt "{CKPT}" --out models/cropguard.onnx --quantize
!ls -la models/

## 7. Holdout evaluation

Runs both the fp32 and INT8 graphs over the test split. Quantisation is only safe to
deploy if the accuracy drop is negligible - so measure it rather than assume it.

In [ ]:
!python -m cropguard.evaluation.predict --model models/cropguard.onnx \
    --split test --out artifacts/preds_fp32.npz --model-version resnet50-fp32
!python -m cropguard.evaluation.predict --model models/cropguard.int8.onnx \
    --split test --out artifacts/preds_int8.npz --model-version resnet50-int8

In [ ]:
from cropguard.evaluation.predict import load_predictions
from cropguard.evaluation.hypothesis import compare_models

fp32 = load_predictions('artifacts/preds_fp32.npz')
int8 = load_predictions('artifacts/preds_int8.npz')
acc32, acc8 = fp32['correct'].mean(), int8['correct'].mean()

print(f'fp32 accuracy     : {acc32:.4f}')
print(f'int8 accuracy     : {acc8:.4f}')
print(f'quantisation drop : {acc32 - acc8:+.4f}')
print()
# A *significant* result here is a reason not to ship INT8, not a curiosity.
print(compare_models(fp32['correct'], int8['correct'], labels=fp32['labels']).summary())

In [ ]:
# macro-F1 is the metric that matters under 36x class imbalance - accuracy hides the
# rare diseases almost entirely.
import json
from sklearn.metrics import f1_score, classification_report

classes = json.load(open('configs/classes.json'))
y_true, y_pred = fp32['labels'], fp32['predictions']
print('accuracy :', (y_true == y_pred).mean())
print('macro-F1 :', f1_score(y_true, y_pred, average='macro'))
print()
print(classification_report(y_true, y_pred, target_names=classes, digits=3, zero_division=0))

## 8. Publish to HuggingFace Hub

Uploads both ONNX graphs, the class list, and a model card. The card is generated from the
prediction files rather than hand-written, so its numbers cannot drift from what the model
actually produced.

Needs a **write**-scoped token from https://hf.co/settings/tokens - a read token fails.
`getpass` keeps it out of the saved notebook.

This is also what the deployed API pulls from at startup, via `CROPGUARD_HF_REPO`.

In [ ]:
HF_REPO_ID = 'XiElonMAsk/cropguard-models'   # <- your HuggingFace namespace

%run notebooks/upload_to_hf.py

---
## 9. OPTIONAL - leakage ablation (~45 min)

Retrains the identical model on a naive stratified split, where 74.2% of test images share
a physical leaf with training. The accuracy gap is the inflation that split causes.

**Warning:** this overwrites `splits.json`. Re-run the split cell in section 3 afterwards
to restore the grouped split before doing anything else.

In [ ]:
!python -m cropguard.data.split --config configs/base.yaml --strategy stratified

import json
print(json.load(open(DATA + '/split_report.json'))['leakage']['test'])

In [ ]:
config = f'''extends: resnet50_baseline.yaml

experiment_name: resnet50-baseline-naive-split

data:
  num_workers: {workers}
'''
Path('configs/colab_resnet50_leaky.yaml').write_text(config)
print(config)

In [ ]:
!python -m cropguard.training.train --config configs/colab_resnet50_leaky.yaml

In [ ]:
print('grouped-split accuracy :', acc32)
print('naive-split   accuracy : see test_acc from the run above')
print()
print('The gap is the accuracy inflation caused by leaf leakage.')
print()
# NOTE: these are different test sets, so this is a descriptive comparison. McNemar needs
# a shared holdout and does not apply across the two splits.

---
### Next

1. Report accuracy, macro-F1, and the fp32 vs INT8 comparison.
2. Deploy: Render reads `render.yaml` and pulls the weights from HF Hub.
3. Train the ConvNeXt-Tiny challenger (`configs/convnext_tiny.yaml`) for the first real
   A/B comparison.